# Set-up Torch

In [ ]:
from mattg.device import device

device = device()

# Set-up MNIST dataset

In [ ]:
from mattg.datasets.bmnist import binarised_mnist

mnist = binarised_mnist()

# Visualise some MNIST samples

In [ ]:
from mattg.plotting import plot

plot(mnist)

# Set-up our (autoregressive) model

In general, we want to estimate the joint probability density function
$$
p(x_1, \ldots, x_n).
$$

The chain rule for probability gives
$$
p(x_1, \ldots, x_{n}) = p(x_1) \, p(x_2 \,|\, x_1) \,\cdots\, p(x_n \,|\, x_{n-1}, \ldots, x_1).
$$

The above problem is intractible, so we need to make some assumptions.

Let's assume a raster-scan ordering of our variables from top-left $X_1$ to bottom-right $X_{n=784}$.

Further, we assume a parameterisation of each conditional probability

$$
p(x_1, \ldots, x_{n}) = p_{\text{CPT}}(x_1; \alpha_1) \, p_{\text{logit}}(x_2 \,|\, x_1; \alpha_2) \,\cdots\, p_{\text{logit}}(x_n \,|\, x_{n-1}, \ldots, x_1; \alpha_n),
$$
where
$$
P_{\text{CPT}}(X_1 = 1; \alpha_1) = \alpha_1, \; P_{\text{CPT}}(X_1 = 0; \alpha_1) = 1 - \alpha_1,
$$
$$
P_{\text{logit}}(X_2 = 1 \,|\, x_1; \alpha_2) = \sigma(\alpha_0^2 + \alpha_1^2x_1)
$$
and $\sigma$ is the softmax function.

This is a Fully Visible Sigmoid Belief Network (FVSBN).

In [ ]:
from mattg.models.rnn import RNN

model_kwargs = {"n_hidden": 28*28//2}
model = RNN(**model_kwargs).to(device)

# Training

# Training a single class

Let's train a single digit

In [ ]:
train_data = mnist

#DIGIT = 3
#indices = [i for i, (_, y) in enumerate(mnist) if y == DIGIT]
#train_data = Subset(mnist, indices)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
criterion = nn.BCEWithLogitsLoss()
optim = torch.optim.Adam(model.parameters(), lr=1e-3)

n_epochs = 5
train_losses = []

for epoch in range(n_epochs):
    model.train()
    running_loss = 0.0
    n_examples = 0

    for x, _ in train_loader:
        # The flattened binary image ARE the target probabilities.
        x = x.to(device).view(x.size(0), -1)

        h = model.h_init(batch_size=x.size(0), device=device)
        loss = 0.0

        for t in range(1, x.size(1)):
            x_t = x[:, t-1:t]
            target_t = x[:, t:t+1]

            logits, h = model(x_t, h)
            loss = loss + criterion(logits, target_t)

        optim.zero_grad()
        loss.backward()
        optim.step()

        running_loss += loss.item() * x.size(0)
        n_examples += x.size(0)

    epoch_loss = running_loss / n_examples
    train_losses.append(epoch_loss)
    print(f"epoch {epoch + 1}/{n_epochs} loss={epoch_loss:.4f}")

# Generate some samples

# Training loss

In [ ]:
import matplotlib.pyplot as plt

plt.plot(range(1, n_epochs + 1), train_losses, marker="o")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Training loss over epochs")
plt.grid(True)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

from mattg.sampling.autoregressive import ancestral_sample

result = ancestral_sample(model, device=device, n_samples=64)
fig, axes = plt.subplots(8, 8, figsize=(12, 12))

for ax, img, probs in zip(axes.flat, result.samples, result.probabilities):
    left = img.detach().cpu().view(28, 28)
    right = probs.detach().cpu().view(28, 28)

    panel = torch.cat([left, right], dim=1)

    ax.imshow(panel, cmap="gray")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from mattg.models.io import save_model

save_model(model, "nade.pt", "NADE", init_kwargs=model_kwargs)